# Feature Engineering

The preprocessing stage cleaned the raw dataset by handling data quality issues such as missing values, incorrect data types, and inconsistent values.

In this notebook, we will transform the cleaned data into meaningful features that can be used by machine learning models.

The goal is not to create as many features as possible. Instead, every transformation should have a clear reason based on the structure of the data and the patterns identified during EDA.

### Main steps

1. Load the preprocessed dataset
2. Separate features and target
3. Remove unsuitable or redundant columns
4. Create meaningful derived features
5. Encode categorical features
6. Check the resulting feature set
7. Save the final dataset for model training

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/telco_cleaned.csv")
df.head()

,City,Zip Code,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,...,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value
0,Los Angeles,90003,33.964131,-118.272783,Male,No,No,No,2,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
1,Los Angeles,90005,34.059281,-118.307420,Female,No,No,Yes,2,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
2,Los Angeles,90006,34.048013,-118.293953,Female,No,No,Yes,8,Yes,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1
3,Los Angeles,90010,34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,...,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1
4,Los Angeles,90015,34.039224,-118.266293,Male,No,No,Yes,49,Yes,...,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1


In [2]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
df.dtypes

Shape: (7043, 24)

Columns:
['City', 'Zip Code', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Value']

Data Types:


City                     str
Zip Code               int64
Latitude             float64
Longitude            float64
Gender                   str
Senior Citizen           str
Partner                  str
Dependents               str
Tenure Months          int64
Phone Service            str
Multiple Lines           str
Internet Service         str
Online Security          str
Online Backup            str
Device Protection        str
Tech Support             str
Streaming TV             str
Streaming Movies         str
Contract                 str
Paperless Billing        str
Payment Method           str
Monthly Charges      float64
Total Charges        float64
Churn Value            int64
dtype: object

In [3]:
df["Churn Value"].value_counts()

Churn Value
0    5174
1    1869
Name: count, dtype: int64

## 1. Separating Target and Features

The objective of this project is to predict whether a customer will churn.

Therefore, `Churn Value` is the target variable and should not be used as an input feature.

Keeping the target separate also helps prevent accidental target leakage during feature engineering.

In [4]:
X = df.drop(columns=["Churn Value"])
y = df["Churn Value"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (7043, 23)
Target shape: (7043,)


## 2. Feature Selection

Before creating new features, we first evaluate whether the existing columns are useful and appropriate for machine learning.

A feature may be removed for several reasons:

- It does not provide meaningful predictive information.
- It is redundant with another feature.
- It has very high cardinality and may cause unnecessary model complexity.
- It represents geographic information that can lead to overfitting.
- It duplicates information already represented by another feature.

The goal is not to remove features simply because they are correlated. A feature will be removed only when there is a clear reason to do so.

### Geographic Features

The dataset contains four geographic variables: `City`, `Zip Code`, `Latitude`, and `Longitude`.

During EDA, strong relationships were observed between `Zip Code`, `Latitude`, and `Longitude`. These variables largely describe the same geographic information from different representations.

Keeping all three would introduce unnecessary redundancy.

`City` is retained because the EDA showed noticeable differences in churn rates across cities. Its usefulness will be evaluated during modeling.

Therefore:

- `Zip Code` → Removed
- `Latitude` → Removed
- `Longitude` → Removed
- `City` → Retained for now

In [5]:
# Dropping Zip Code, Latitude, and Longitude because they are strongly related to City and to each other.
X = X.drop(columns=["Zip Code", "Latitude", "Longitude"])
X.shape

(7043, 20)

### Total Charges

`Total Charges` has a strong relationship with `Tenure Months` and a moderate-to-strong relationship with `Monthly Charges`.

Although this makes it partially redundant, it is not target leakage because the value is available at prediction time.

Therefore, `Total Charges` will initially be retained. Its contribution will be evaluated during model training and feature importance analysis.


In [6]:
print("Features remaining:", X.shape[1])
print(X.columns.tolist())

Features remaining: 20
['City', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges']


## 3. Feature Creation

Now that the redundant geographic columns are gone, I'll create a few new features based on things that stood out during EDA.

### Avg Monthly Spend

`Total Charges` is basically the accumulation of `Monthly Charges` over `Tenure Months`, so dividing the two gives a rough average monthly spend for each customer.

For customers with `Tenure Months = 0`, this would be a division by zero, so I fall back to `Monthly Charges` for those rows since that's the only spend info we have for them anyway.

In [7]:
X["Avg Monthly Spend"] = np.where(
    X["Tenure Months"] > 0,
    X["Total Charges"] / X["Tenure Months"],
    X["Monthly Charges"]
)

X[["Tenure Months", "Total Charges", "Monthly Charges", "Avg Monthly Spend"]].head()

,Tenure Months,Total Charges,Monthly Charges,Avg Monthly Spend
0,2,108.15,53.85,54.075000
1,2,151.65,70.70,75.825000
2,8,820.50,99.65,102.562500
3,28,3046.05,104.80,108.787500
4,49,5036.30,103.70,102.781633


### Num Additional Services

There are 6 add-on service columns (`Online Security`, `Online Backup`, `Device Protection`, `Tech Support`, `Streaming TV`, `Streaming Movies`). Instead of only encoding each one separately, I'm also creating a single count feature that tells us how many of these a customer has subscribed to. This gives the model one aggregate signal for "how attached is this customer to our ecosystem" alongside the individual service columns.

In [8]:
service_cols = [
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies"
]

X["Num Additional Services"] = (X[service_cols] == "Yes").sum(axis=1)

X["Num Additional Services"].value_counts().sort_index()

Num Additional Services
0    2219
1     966
2    1033
3    1118
4     852
5     571
6     284
Name: count, dtype: int64

### Tenure Group

EDA showed tenure has a strong relationship with churn, so grouping it into broad lifecycle stages might help the model pick up on that pattern more directly. I'm keeping the original `Tenure Months` column too, this is just an extra grouped view of it, not a replacement.

Groups:
- 0-12 -> New
- 13-24 -> Short-term
- 25-48 -> Mid-term
- 49-72 -> Long-term

In [9]:
X["Tenure Group"] = pd.cut(
    X["Tenure Months"],
    bins=[-1, 12, 24, 48, 72],
    labels=["New", "Short-term", "Mid-term", "Long-term"]
)

X["Tenure Group"].value_counts()

Tenure Group
Long-term     2239
New           2186
Mid-term      1594
Short-term    1024
Name: count, dtype: int64

In [10]:
print("Features after creation step:", X.shape[1])
X.columns.tolist()

Features after creation step: 23


['City',
 'Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Avg Monthly Spend',
 'Num Additional Services',
 'Tenure Group']

## 4. Encoding Categorical Features

Before encoding anything I want to split the categorical columns into groups, because they don't all need the same treatment:

- **Binary columns** (only 2 categories, like Yes/No or Male/Female) -> map to 0/1 directly, no need for one-hot here since there's no ordering issue with only 2 values.
- **Multi-category nominal columns** (3-4 categories, like Contract or Payment Method) -> one-hot encode, since these are small enough that one-hot won't blow up the feature space.
- **City** -> needs its own treatment, handled separately below, since it has 1000+ unique values.

In [11]:
binary_cols = [
    "Gender",
    "Senior Citizen",
    "Partner",
    "Dependents",
    "Phone Service",
    "Paperless Billing"
]

multi_category_cols = [
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Contract",
    "Payment Method",
    "Tenure Group"
]

print("Binary cols:", len(binary_cols))
print("Multi-category cols:", len(multi_category_cols))

Binary cols: 6
Multi-category cols: 11


### Binary Encoding

In [12]:
# Gender is Male/Female, so it needs its own mapping
X["Gender"] = X["Gender"].map({"Male": 1, "Female": 0})

# The rest are all Yes/No
for col in ["Senior Citizen", "Partner", "Dependents", "Phone Service", "Paperless Billing"]:
    X[col] = X[col].map({"Yes": 1, "No": 0})

X[binary_cols].head()

,Gender,Senior Citizen,Partner,Dependents,Phone Service,Paperless Billing
0,1,0,0,0,1,1
1,0,0,0,1,1,1
2,0,0,0,1,1,1
3,0,0,1,1,1,1
4,1,0,0,1,1,1


### One-Hot Encoding the Multi-Category Columns

Using `drop_first=True` here so we don't keep a redundant column for each feature (e.g. if it's not DSL and not "No" internet, it has to be Fiber optic, so we don't need a separate column for that).

In [13]:
X = pd.get_dummies(X, columns=multi_category_cols, drop_first=True, dtype=int)

print("Shape after one-hot encoding:", X.shape)

Shape after one-hot encoding: (7043, 36)


### Handling City

City has over a thousand unique values, so one-hot encoding it directly would add over a thousand mostly-empty columns, which isn't practical here.

Target encoding (using churn rate per city) would work in theory, but doing it correctly requires fitting it on a training split only, otherwise the target information leaks into the feature. Since this notebook doesn't do the train/test split (that happens in modeling), I'm not doing target encoding here.

Instead I'm using frequency encoding, replacing each city with how many customers come from that city. It doesn't touch the target at all so there's no leakage risk, and it still gives the model some signal about the city instead of throwing the column away completely.

In [14]:
city_counts = X["City"].value_counts()
X["City Freq"] = X["City"].map(city_counts)

X = X.drop(columns=["City"])

X[["City Freq"]].describe()

,City Freq
count,7043.000000
mean,30.291211
std,65.876109
min,4.000000
25%,4.000000
50%,5.000000
75%,16.000000
max,305.000000


## 5. Final Feature Matrix Check

In [15]:
print("Final feature matrix shape:", X.shape)
X.dtypes.value_counts()

Final feature matrix shape: (7043, 36)


int64      33
float64     3
Name: count, dtype: int64

In [16]:
X.columns.tolist()

['Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Paperless Billing',
 'Monthly Charges',
 'Total Charges',
 'Avg Monthly Spend',
 'Num Additional Services',
 'Multiple Lines_No phone service',
 'Multiple Lines_Yes',
 'Internet Service_Fiber optic',
 'Internet Service_No',
 'Online Security_No internet service',
 'Online Security_Yes',
 'Online Backup_No internet service',
 'Online Backup_Yes',
 'Device Protection_No internet service',
 'Device Protection_Yes',
 'Tech Support_No internet service',
 'Tech Support_Yes',
 'Streaming TV_No internet service',
 'Streaming TV_Yes',
 'Streaming Movies_No internet service',
 'Streaming Movies_Yes',
 'Contract_One year',
 'Contract_Two year',
 'Payment Method_Credit card (automatic)',
 'Payment Method_Electronic check',
 'Payment Method_Mailed check',
 'Tenure Group_Short-term',
 'Tenure Group_Mid-term',
 'Tenure Group_Long-term',
 'City Freq']

### Checking for Missing / Infinite Values

Just making sure none of the transformations above introduced anything weird before saving this.

In [17]:
print("Missing values:", X.isnull().sum().sum())
print("Infinite values:", np.isinf(X.select_dtypes(include=[np.number])).sum().sum())

Missing values: 0
Infinite values: 0


No missing or infinite values, so the feature matrix is clean.

## 6. Save the Engineered Dataset

Saving `X` and `y` together so the modeling notebook can load one file and split them back out. This keeps everything from this notebook in one place for the next stage.

In [18]:
final_df = X.copy()
final_df["Churn Value"] = y

final_df.to_csv("../data/processed/telco_features.csv", index=False)

print("Saved:", final_df.shape)

Saved: (7043, 37)


## Feature Engineering Summary

Starting from the 23 cleaned input features, I:

- Dropped `Zip Code`, `Latitude`, and `Longitude` for being redundant with each other and with `City`.
- Kept `City` and `Total Charges` for now, both are potentially useful and neither is a leakage risk.
- Created `Avg Monthly Spend`, `Num Additional Services`, and `Tenure Group` based on patterns seen during EDA.
- Binary-encoded the 2-category columns and one-hot encoded the small multi-category columns.
- Frequency-encoded `City` instead of one-hot encoding or target encoding it, to avoid both feature explosion and leakage.

This gives a clean numeric feature matrix ready for the model-training notebook.